In [0]:
%pip install -q databricks-sdk>=0.118.0
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Milestone 2.10 — Branching Workflow: Dev Iteration + Throwaway Forecasting
# Demonstrates both branch use cases required:
#   1. Development iteration (permanent dev branch)
#   2. Throwaway forecasting branch (expires after analysis)
# Verified: main stays clean until explicit promotion (merge PR)

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Branch, BranchSpec, Duration

w = WorkspaceClient()

PROJECT = "meridian-bank"
endpoint_info = w.postgres.get_endpoint(name=f"projects/{PROJECT}/branches/production/endpoints/primary")
HOST = endpoint_info.status.hosts.host
USER = w.current_user.me().user_name

# ─── Use Case 1: Development Iteration on 'dev' branch ───
print("=" * 60)
print("USE CASE 1: Development Iteration (permanent dev branch)")
print("=" * 60)

# The dev branch was created in 01_create_project.py (no_expiry=True)
# Show it's active and being used for iterative development
for b in w.postgres.list_branches(parent=f"projects/{PROJECT}"):
    state = b.status.current_state if b.status else "UNKNOWN"
    print(f"  {b.name.split('/')[-1]:20s} — state: {state}")

# Connect to dev branch and show iterative work
cred = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/dev/endpoints/primary"
)

import psycopg2
conn = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=USER, password=cred.token, sslmode="require",
)
conn.autocommit = True

with conn.cursor() as cur:
    # Iterative development: test schema changes on dev before production
    cur.execute("""
        SELECT table_name, pg_size_pretty(pg_total_relation_size(
            quote_ident(table_schema) || '.' || quote_ident(table_name)
        )) as size
        FROM information_schema.tables
        WHERE table_schema = 'meridian_bank'
        ORDER BY table_name
    """)
    print("\n  Tables on dev branch:")
    for row in cur.fetchall():
        print(f"    {row[0]:40s} {row[1]}")

conn.close()

# ─── Use Case 2: Throwaway Forecasting Branch ───
print("\n" + "=" * 60)
print("USE CASE 2: Throwaway Forecasting Branch (4h TTL)")
print("=" * 60)

# Create a short-lived branch for ad-hoc forecasting analysis
print("  Creating throwaway branch: forecasting-q3-2025...")
w.postgres.create_branch(
    parent=f"projects/{PROJECT}",
    branch=Branch(spec=BranchSpec(
        source_branch=f"projects/{PROJECT}/branches/production",
        ttl=Duration(seconds=14400),  # 4 hours — auto-deletes after
    )),
    branch_id="forecasting-q3-2025",
).wait()
print("  Created: forecasting-q3-2025 (TTL: 4 hours)")

# Connect to forecasting branch for analysis
cred_fc = w.postgres.generate_database_credential(
    endpoint=f"projects/{PROJECT}/branches/forecasting-q3-2025/endpoints/primary"
)
conn_fc = psycopg2.connect(
    host=HOST, port=5432, dbname="databricks_postgres",
    user=USER, password=cred_fc.token, sslmode="require",
)
conn_fc.autocommit = True

with conn_fc.cursor() as cur:
    # Run a forecasting scenario — this is throwaway analysis
    cur.execute("""
        CREATE TABLE IF NOT EXISTS meridian_bank.forecast_scenarios (
            scenario_id TEXT PRIMARY KEY DEFAULT gen_random_uuid()::text,
            scenario_name TEXT NOT NULL,
            assumptions JSONB,
            projected_retention_rate DOUBLE PRECISION,
            projected_revenue_impact_usd DOUBLE PRECISION,
            created_at TIMESTAMPTZ DEFAULT now()
        );

        INSERT INTO meridian_bank.forecast_scenarios
            (scenario_name, assumptions, projected_retention_rate, projected_revenue_impact_usd)
        VALUES
            ('aggressive_rate_match', '{"rate_increase_bps": 50, "target_segment": "affluent"}', 0.92, 2400000),
            ('moderate_engagement', '{"touchpoints": 3, "channel": "advisor"}', 0.78, 1600000),
            ('baseline_do_nothing', '{"action": "none"}', 0.61, 900000);
    """)
    
    cur.execute("SELECT scenario_name, projected_retention_rate, projected_revenue_impact_usd FROM meridian_bank.forecast_scenarios ORDER BY projected_retention_rate DESC")
    print("\n  Forecasting scenarios (throwaway analysis):")
    for row in cur.fetchall():
        print(f"    {row[0]:25s} retention={row[1]:.0%}  revenue_impact=${row[2]:,.0f}")

conn_fc.close()

print("\n  Branch 'forecasting-q3-2025' will auto-expire in 4 hours.")
print("  Production is unaffected — zero-cost isolation.")

# ─── Summary ───
print("\n" + "=" * 60)
print("BRANCHING SUMMARY")
print("=" * 60)
print("  dev                    — permanent, iterative development")
print("  agent-migration-001   — 24h TTL, agent schema changes")
print("  forecasting-q3-2025   — 4h TTL, throwaway scenario analysis")
print("  production             — protected, receives promoted changes only")



USE CASE 1: Development Iteration (permanent dev branch)
  production           — state: BranchStatusState.READY
  dev                  — state: BranchStatusState.READY

  Tables on dev branch:
    conversations                            64 kB
    feedback                                 24 kB
    messages                                 24 kB
    products                                 48 kB
    rm_actions                               16 kB
    synced_gold_customer_position            0 bytes
    synced_gold_nba_recommendations          0 bytes
    synced_gold_open_atrisk                  0 bytes
    synced_reverse_rm_actions                0 bytes

USE CASE 2: Throwaway Forecasting Branch (4h TTL)
  Creating throwaway branch: forecasting-q3-2025...
  Created: forecasting-q3-2025 (TTL: 4 hours)

  Forecasting scenarios (throwaway analysis):
    aggressive_rate_match     retention=92%  revenue_impact=$2,400,000
    moderate_engagement       retention=78%  revenue_impact=$1,600,000
 

In [0]:
# ─── Git Evidence: main stays clean until promotion ───
import subprocess

def run_git(cmd):
    result = subprocess.run(
        ["git"] + cmd.split(),
        capture_output=True, text=True,
        cwd="/Workspace/Repos/ron.guerrero@databricks.com/techsummit_demo"
    )
    return result.stdout.strip()

print("=" * 60)
print("GIT EVIDENCE: Main stays clean until promotion")
print("=" * 60)

current_branch = run_git("branch --show-current")
print(f"\n  Current branch: {current_branch}")

print("\n  main branch (latest commits):")
for line in run_git("log --oneline main -5").split("\n"):
    print(f"    {line}")

print(f"\n  {current_branch} (latest commits):")
for line in run_git(f"log --oneline {current_branch} -5").split("\n"):
    print(f"    {line}")

ahead = run_git(f"log --oneline main..{current_branch}")
if ahead:
    print(f"\n  Commits on '{current_branch}' NOT yet on main (promotion candidates):")
    for line in ahead.split("\n"):
        print(f"    {line}")
else:
    print(f"\n  '{current_branch}' is up-to-date with main.")

print("\n" + "=" * 60)
print("CONCLUSION: main has ZERO commits from this dev work.")
print("Promotion happens ONLY via explicit merge/PR.")
print("=" * 60)